# Multilayer Perceptron Neural Network Architecture for Network Anomaly Detection
## Group 5 — Member 3 (Model Lead): Implementation, Ablation, and Experimental Benchmarking
---
This notebook provides the complete, executable pipeline for:
1. **Data Preprocessing**: Strict training-fitted StandardScaler and OneHotEncoder without data leakage.
2. **4-Layer Feed-Forward MLP**: Progressive dimensional compression `[122 -> 64 -> 32 -> 16 -> 1]`.
3. **Custom Adam LR Decay**: 100-epoch custom training loop with stepwise learning-rate decay.
4. **Controlled Ablation (RQ4)**: Evaluating the necessity of LR decay against a fixed-rate model.
5. **Baseline Comparison (RQ2)**: Benchmarking against Logistic Regression and Decision Tree.
6. **Evaluation & Visualization**: Confusion matrices, ROC curves, Loss histories, and Class-wise metrics.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_pipeline import prepare_data
from src.model import (
    train_mlp_custom_schedule,
    train_mlp_fixed_schedule,
    train_baseline_logistic_regression,
    train_baseline_decision_tree
)
from src.evaluate import (
    evaluate_model,
    plot_loss_curves,
    plot_learning_rate_schedule,
    plot_confusion_matrices,
    plot_roc_curves,
    plot_classwise_metrics,
    plot_architecture_diagram
)

print('Environment and modules loaded successfully!')

### 1. Data Loading & Preprocessing
Loads `train_features.csv`, `validation_features.csv`, and `test_features.csv`. All transformers fit strictly on `X_train`.

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test, preprocessor, meta = prepare_data()
print(f'Training shape:   {X_train.shape} | Labels: {y_train.shape}')
print(f'Validation shape: {X_val.shape}   | Labels: {y_val.shape}')
print(f'Testing shape:    {X_test.shape}  | Labels: {y_test.shape}')

### 2. Train Proposed 4-Layer MLP with Custom Adam LR Decay (100 Epochs)
Progressively decays learning rate at epochs 20, 40, 60, and 80.

In [ ]:
mlp_decay, decay_history = train_mlp_custom_schedule(
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    epochs=100, random_state=42, verbose=True
)

### 3. Train Ablation Fixed-Rate MLP (Constant LR = 0.001)
Addresses **RQ4** by controlling for optimization dynamics.

In [ ]:
mlp_fixed, fixed_history = train_mlp_fixed_schedule(
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    epochs=100, learning_rate=0.001, random_state=42, verbose=False
)
print('Ablation fixed-rate model finished training.')

### 4. Train Baselines (Logistic Regression & Decision Tree)
Addresses **RQ2** with identical standardized inputs.

In [ ]:
lr_model, lr_time = train_baseline_logistic_regression(X_train, y_train, random_state=42)
dt_model, dt_time = train_baseline_decision_tree(X_train, y_train, max_depth=10, random_state=42)
print(f'Logistic Regression fit in {lr_time:.2f}s')
print(f'Decision Tree fit in {dt_time:.2f}s')

### 5. Final Test Set Evaluation & Comparative Analysis
Evaluates performance on unseen `KDDTest+` (22,544 connections).

In [ ]:
models = [
    ('Proposed MLP (Custom LR Decay)', mlp_decay, decay_history['total_training_time_sec']),
    ('Ablation MLP (Fixed LR 0.001)', mlp_fixed, fixed_history['total_training_time_sec']),
    ('Baseline (Logistic Regression)', lr_model, lr_time),
    ('Baseline (Decision Tree)', dt_model, dt_time)
]

eval_results = []
rows = []
for name, model, t_train in models:
    res = evaluate_model(model, X_test, y_test, model_name=name)
    res['training_time_sec'] = t_train
    eval_results.append(res)
    rows.append({
        'Model': name,
        'Accuracy': round(res['accuracy'], 4),
        'Precision (Attack)': round(res['precision'], 4),
        'Recall (Attack)': round(res['recall'], 4),
        'F1-Score': round(res['f1_score'], 4),
        'ROC-AUC': round(res['roc_auc'], 4),
        'False Alarm Rate (FPR)': round(res['false_positive_rate'], 4),
        'Miss Rate (FNR)': round(res['false_negative_rate'], 4),
        'Training Time (s)': round(t_train, 2)
    })

df_results = pd.DataFrame(rows)
df_results

### 6. Visualizations
Display the generated publication figures.

In [ ]:
from IPython.display import Image, display
figures = [
    'outputs/figures/loss_curve.png',
    'outputs/figures/lr_schedule.png',
    'outputs/figures/confusion_matrices.png',
    'outputs/figures/roc_curves.png',
    'outputs/figures/classwise_metrics.png',
    'outputs/figures/architecture_diagram.png'
]
for f in figures:
    if Path(f).exists():
        print(f'Displaying: {f}')
        display(Image(filename=f))
